# koilog-health — нагрузка, часы Apple Watch и болезни

`koilog.health` связывает внешнюю нагрузку (журнал силовых) с внутренней
(ночной пульс, ВСР, дыхание, температура запястья, сон из Apple Watch) и
находит, как расбаланс предшествует эпизодам болезни. Болезнь здесь — исход,
который связывается с нагрузкой, а не диагноз.

Вся логика живёт в пакете `koilog.health` и вызывается из командной строки;
этот ноутбук — обзор поверх той же базы. Он импортирует **те же** функции, что
и CLI, поэтому расхождений между выводом здесь и в терминале нет.

```
python -m koilog apple-load        # разобрать выгрузку Apple Health в aw_*
python -m koilog health-load       # внешняя нагрузка: load_daily, health_days
python -m koilog illness           # метки эпизодов: confirmed/probable/gap_only
python -m koilog ml                # признаки, CV-отчёт, красный флаг
python -m koilog ml --lag          # + лаг-анализ признаков
```

Пакет держится на стандартной библиотеке. `matplotlib` нужен только этому
ноутбуку (extra `notebook`), `numpy`/`scikit-learn` — только разделу ML
(extra `ml`). Без них остальные разделы работают.

## Как читать числа

Короткий словарь, чтобы колонки и метрики не приходилось угадывать.

**Тоннаж** — суммарный кг за день: Σ (вес × повторы × подходы). **sRPE** —
суррогат тренировочного стресса (тоннаж × целевое RPE); здесь он нормирован
и служит «острой» нагрузкой. **ACWR** — отношение острой (7 дней) к
хронической (28 дней) нагрузке: ~1 — баланс, >1,3 — риск перегруза.

**Монотонность** — среднедневная нагрузка, делённая на её дневной SD за
неделю: чем ровнее и однообразнее неделя, тем выше. **Стрейн** — недельная
нагрузка × монотонность (Foster). **e1RM** — оценка максимума по последнему
подходу; «просадка e1RM» — бинарный флаг регресса.

**Эпизод болезни** — склейка соседних дней с меткой. **`confirmed`** — ручная
температура ≥37,5 °C либо ≥2 нарушенных каналов за день. **`probable`** —
одиночный сильный сигнал автономного ядра (RHR ↑ или ВСР ↓ при z>3). ЧДД и
температура запястья в одиночку эпизод не открывают — они лишь подтверждают.
**`gap_only`** — только перерыв в тренировках ≥14 дней, без физиологии; в
обучение не идёт и отчитывается отдельно.

**z-скор** — отклонение канала от личного фона за предыдущие 60 дней (в SD);
фон — строго прошлые дни, текущий не входит. **Долг сна** — сколько часов за
7 дней недоспал относительно своего обычного сна.

**Красный флаг** (Meeusen) — ≥2 из 4 признаков за 14 дней до эпизода: просадка
e1RM, ВСР ниже обычного, долг сна, монотонность >2. **recall эпизода** — доля
эпизодов, у которых был хотя бы один флагированный день. **ложн./мес** —
сколько дней в месяц модель окажется ложной.

In [ ]:
# Подключение к уже разобранной базе. Таблицы aw_*/load_daily/ml_* должны
# существовать: сначала `python -m koilog apple-load && health-load && illness`.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

from koilog import etl
from koilog.health import analysis, apple, illness, load, ml, text

conn = etl.connect(etl.DEFAULT_DB)
print(etl.DEFAULT_DB)

## Покрытие часов по годам

Метрики появляются в выгрузке Apple Watch не все сразу и не с самого начала.
Ниже — сколько дней в каждом году реально содержит каждую метрику. Полезное
окно задано константой `apple.DEFAULT_WINDOW` (2024-01-16): раньше часы пишут
только пульс покоя и сон, без ВСР/ЧДД/температуры, поэтому они в анализ не идут.

In [ ]:
# Покрытие метрик по календарным годам. То же: `python -m koilog apple-load`.
print(apple.format_coverage_by_year(apple.coverage_by_year(conn)))
print()
# Сводное покрытие за окно: сколько дней заполнено каждой метрикой.
print(apple.format_coverage(apple.coverage(conn)))

## Внешняя нагрузка

`health-load` строит непрерывный календарь нагрузки: дни отдыха — нули, а не
пропуски. Ниже сводка тоннажа, sRPE, ACWR, монотонности и стрейна за окно.

In [ ]:
# Сводка нагрузки за окно. То же печатает `python -m koilog health-load`.
print(load.format_summary(conn))

## Симптомная сигнатура

Средние каналов внутренней нагрузки внутри каждого подтверждённого эпизода и в
обычные дни (фон). Это картина гриппоподобной реакции: ночной пульс ↑, ВСР ↓,
ЧДД ↑, температура запястья ↑, сон ↓ (Mishra 2020, Quer 2021, Radin 2020).
Последняя строка — фон, с которым всё сравнивается.

In [ ]:
# Сигнатура подтверждённых эпизодов против фона.
rows = analysis.signature(conn)
headers = ["с", "по", "дней", "уровень",
           "RHR", "ВСР", "ЧДД", "t запястья", "сон"]
print(text.table(headers, [
    [r["start"], r["end"], r["days"], r["level"],
     r["RHR, уд/мин"], r["ВСР, мс"], r["ЧДД"],
     r["t запястья, °C"], r["сон, ч"]]
    for r in rows]))

## Эпизоды болезни

Многоуровневые метки. `gap_only` в обучение не идёт и ложится в отчёт
отдельно — «только перерыв» это не болезнь, а отсутствие тренировок.

In [ ]:
# Список эпизодов. То же печатает `python -m koilog illness`.
print(illness.format_episodes(conn))

## Спагетти: 28 дней до каждого эпизода

Каждая линия — один эпизод, выровненный по началу болезни (лаг 0 — день до
начала). Тонкие синие — эпизоды, жирная красная — среднее по ним; серые —
контрольные окна тех же размеров из дней, далёких от любой болезни, и их
среднее. Расхождение красной и серой у лага 0 — предболезненный сигнал; без
контрольных окон его нельзя отличить от обычной шумности признака.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Четыре признака: ACWR, монотонность, долг сна и ВСР (z-к своему фону).
FEATURES = [
    ("acwr", "ACWR (sRPE 7/28)"),
    ("monotony", "монотонность недели"),
    ("sleep_debt7", "долг сна, ч"),
    ("z_hrv", "ВСР, z-скор"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (feature, title) in zip(axes.ravel(), FEATURES):
    block = analysis.pre_episode_windows(conn, feature)
    lags = block["lags"]
    for row in block["control"]:
        ax.plot(lags, row, color="0.75", lw=0.8)
    for row in block["series"]:
        ax.plot(lags, row, color="#4c72b0", lw=1.2, alpha=0.6)
    ax.plot(lags, block["control_mean"], color="0.3", lw=2.2, label="контроль")
    ax.plot(lags, block["mean"], color="#c44e52", lw=2.4,
            label=f"эпизоды (n={block['n']})")
    ax.set_title(title)
    ax.set_xlabel("дней до начала эпизода")
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

## Лаг-анализ

Для каждого лага 0–21 день до начала эпизода — насколько среднее признака (в SD
от его личного фона) отклонено относительно всех остальных дней. Это
описательная величина, **без p-value**: при ~30 эпизодах p-value был бы
самообманом.

In [ ]:
# Лаг-анализ тоннажа и ВСР. То же с ключом --lag: `python -m koilog ml --lag`.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, feature in zip(axes, ("weekly_tonnage", "z_hrv")):
    rows = ml.lag_analysis(conn, feature)
    print(ml.format_lag(rows, feature))
    lags = [r["lag"] for r in rows]
    effect = [r["effect"] if r["effect"] is not None else float("nan")
              for r in rows]
    ax.axhline(0, color="0.6", lw=1)
    ax.plot(lags, effect, marker="o", ms=3.5, lw=1.6, color="#4c72b0")
    ax.set_title(feature)
    ax.set_xlabel("лаг, дней до начала эпизода")
    ax.set_ylabel("эффект, SD")
fig.tight_layout()
plt.show()

## Персональные квантили: предболезненные дни против обычных

Каналы внутренней нагрузки в днях за 14 дней до эпизода (`y=1` в
`ml_features`) против остальных дней. **сдвиг медианы** — насколько медиана
предболезненных дней сдвинута в SD личного фона относительно обычной медианы.

In [ ]:
rows = analysis.quantile_profile(conn)
headers = ["канал", "обыч q25", "q50", "q75",
           "предб. q25", "q50", "q75", "сдвиг медианы"]
print(text.table(headers, [
    [r["channel"], r["q25"], r["q50"], r["q75"],
     r["q25_pre"], r["q50_pre"], r["q75_pre"], r["median_shift"]]
    for r in rows]))

## ML: CV-отчёт и красный флаг

Протокол — leave-one-episode-out (случайные складки запрещены: временная течь).
**бейзлайн** учитывает только физиологию (без нагрузки), **GBM** — проверка,
стоит ли нелинейная модель своих хлопот.

**Честная оговорка.** Эпизодов ~30 за два года — это не про реальный
предиктор, а про гипотезу. Метрики ниже нужны, чтобы сравнить модели между
собой и найти красный флаг, а не чтобы обещать раннее предупреждение.

In [ ]:
# Полный ML-отчёт (пересобирает признаки и метки). То же: `python -m koilog ml`.
data = ml.report(conn)
print(ml.format_report(data))
print()
# Красный флаг Meeusen: ≥2 признака за 14 дней до эпизода.
flag = ml.red_flag_report(conn)
print(f"красный флаг: эпизодов {flag['episodes']}, "
      f"с флагом {flag['episodes_with_flag']}, "
      f"максимум признаков {flag['max_points']}")

## Дальше: возврат к рабочим весам после болезни

Это цель следующего шага (план раунда 4, §7); кода пока нет. Когда болезнь
прерывает цикл, сила у тренированных почти не падает 2–3 недели (Mujika &
Padilla 2000), но возвращаться надо от последнего «до болезни» рабочего веса, а
не от общего плана. Опорные точки для будущей реализации:

- **кривые детрейна**: e1RM против числа дней простоя;
- **рампа возврата**: % от предболезненного веса на сессию, ограничиваемая КОИ
  и восстановлением ЧСС/ВСР;
- **порог «потеря >2 недель»** (Raysmith & Drew 2016) как ориентир отката;
- **скорость реампера против рецидива**: слишком быстрый возврат → повторный
  эпизод (проверяется на своих же «жёстких» и «мягких» возвратах).

Реализация опирается на существующую логику волны в `koilog.core` и выдаёт план
по упражнению с даты возврата.